# 🌱 Цифровой двойник теплицы — GreenLight

## Что такое цифровой двойник?

**Цифровой двойник** — это виртуальная копия реальной теплицы, которая:
- Моделирует физику и биологию теплицы в реальном времени
- Получает данные с реальных датчиков (температура, влажность, CO₂)
- Сравнивает модель с измерениями и выявляет отклонения
- Позволяет тестировать сценарии управления **без риска для урожая**

```
┌──────────────────────┐         ┌──────────────────────────┐
│   ФИЗИЧЕСКАЯ         │  данные │   ЦИФРОВАЯ МОДЕЛЬ        │
│   ТЕПЛИЦА            │ ──────► │   (GreenLight)           │
│                      │         │                          │
│  🌡 Датчики темп.    │         │  📊 tAir, tCan, tFlr     │
│  💧 Датчики влажн.   │         │  💧 vpAir, RH            │
│  🌬 Датчики CO₂      │         │  🌿 CO₂, фотосинтез      │
│  ☀️ Пиранометр       │         │  🍅 Рост урожая          │
└──────────────────────┘         └──────────────────────────┘
```

---
**Модель**: Katzin 2021 (Вагенингенский университет)  
**Уравнения**: ODE-система из 28 состояний, 189 вспомогательных переменных, 151 параметр

## 1. Установка и импорты

In [ ]:
# Проверяем зависимости
import subprocess, sys

required = ['greenlight', 'numpy', 'pandas', 'matplotlib', 'scipy', 'ipywidgets']
missing = []
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'Устанавливаем: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing)
else:
    print('✅ Все зависимости установлены')

In [ ]:
# Если запускаем из папки проекта, используем установленный пакет
import sys, os
GREENLIGHT_SITE = '/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages'
if GREENLIGHT_SITE not in sys.path:
    sys.path.insert(0, GREENLIGHT_SITE)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ipywidgets для интерактивности
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, Layout, HBox, VBox, Label
from IPython.display import display, clear_output

from greenlight import GreenLight

# Стиль графиков
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#e6edf3',
    'text.color': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'grid.linewidth': 0.8,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'legend.labelcolor': '#e6edf3',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

print('✅ Импорты выполнены')
print(f'📦 GreenLight готов')

## 2. Структура модели GreenLight

Модель разделена на **3 подсистемы**:

| Подсистема | Описание | Состояния |
|---|---|---|
| 🏗 **Климат теплицы** | Теплообмен, вентиляция, экраны | tAir, tCan, tFlr, tSo1-5, tPipe... |
| 🌿 **Рост урожая** | Фотосинтез, распределение ассимилятов | cBuf, cFruit, cLeaf, cStem |
| 🎛 **Управление** | Отопление, вентиляция, освещение | uBoil, uVent, uLamp... |

In [ ]:
# Загружаем модель и смотрим её структуру
mdl = GreenLight()
mdl.load()

print('='*60)
print('📋 СТРУКТУРА МОДЕЛИ GreenLight (Katzin 2021)')
print('='*60)
print(f'\n🔵 Состояния (state variables): {len(mdl.states)}')
for name in mdl.states:
    v = mdl.states[name]
    unit = v.get('unit', '?')
    desc = v.get('description', name)
    print(f'  • {name:15s} [{unit:20s}]  {desc[:45]}')

print(f'\n🟡 Входные данные (inputs): {len(mdl.inputs)}')
for name in mdl.inputs:
    v = mdl.inputs[name]
    unit = v.get('unit', '?')
    desc = v.get('description', name)
    print(f'  • {name:20s} [{unit:15s}]  {desc[:45]}')

print(f'\n🔴 Вспомогательные переменные (aux): {len(mdl.aux)}')
print(f'⚫ Константы (parameters):           {len(mdl.const)}')

## 3. Запуск симуляции

Симуляция решает систему ОДУ (28 уравнений) численным методом BDF.  
**Входные данные**: погода Блейсвейк (Нидерланды), октябрь 2009.  
**Длительность**: 7 дней (604800 секунд).

In [ ]:
%%time
print('🚀 Запускаем симуляцию...')
print('   Метод: BDF (неявный, для жёстких ОДУ)')
print('   Период: 7 дней')

# Симуляция 7 дней
mdl = GreenLight(
    input_prompt=[
        None,  # используем модель по умолчанию
        None,  # используем данные по умолчанию
    ],
    options={
        't_start': '0',
        't_end': str(7 * 24 * 3600),  # 7 дней
        'output_step': '900',          # каждые 15 минут
        'solver': 'BDF',
        'rtol': '1e-4',
        'atol': '1e-2',
    }
)
mdl.load()
mdl.solve()

df = mdl.full_sol.copy()
# Преобразуем время в часы и datetime
t0 = datetime(2009, 10, 20)  # начало данных Блейсвейк
df['hours'] = df['Time'] / 3600
df['dt'] = [t0 + timedelta(seconds=float(s)) for s in df['Time']]

print(f'\n✅ Симуляция завершена!')
print(f'   Точек времени: {len(df)}')
print(f'   Переменных:    {len(df.columns)}')
print(f'   Период:        {df["dt"].iloc[0]} — {df["dt"].iloc[-1]}')

In [ ]:
# Вычислим относительную влажность (не входит в стандартный вывод)
# RH = vpAir / vp_sat(tAir) * 100
# vp_sat(T) = 610.78 * exp(17.27*T / (T+237.3))  [Па]
def vp_sat(T_C):
    return 610.78 * np.exp(17.27 * T_C / (T_C + 237.3))

df['RH_air'] = df['vpAir'] / vp_sat(df['tAir']) * 100
df['RH_air'] = df['RH_air'].clip(0, 100)

# Суммарная биомасса
df['biomass_total'] = df['cFruit'] + df['cLeaf'] + df['cStem']

# Перевод CO₂ из мг/м³ в ppm
if 'co2Air' in df.columns:
    df['co2_ppm'] = df['co2Air'] / 1.96  # мг/м³ → ppm

print('📊 Ключевые показатели за 7 дней:')
print('='*50)

metrics = [
    ('tAir',  'Темп. воздуха',  '°C'),
    ('tCan',  'Темп. кронopy',  '°C'),
    ('RH_air','Влажность',      '%'),
    ('tPipe', 'Темп. труб',     '°C'),
]
if 'co2_ppm' in df.columns:
    metrics.append(('co2_ppm', 'CO₂', 'ppm'))

for col, label, unit in metrics:
    if col in df.columns:
        print(f'  {label:20s}: {df[col].mean():.1f} avg | {df[col].min():.1f} min | {df[col].max():.1f} max  [{unit}]')

print()
print(f'  {"Плоды (конец)":20s}: {df["cFruit"].iloc[-1]/1000:.1f} г·CH₂O/м²')
print(f'  {"Листья (конец)":20s}: {df["cLeaf"].iloc[-1]/1000:.1f} г·CH₂O/м²')
print(f'  {"Буфер (конец)":20s}: {df["cBuf"].iloc[-1]/1000:.1f} г·CH₂O/м²')

## 4. 🌡 Климат теплицы

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
fig.suptitle('🌡 Температурный профиль теплицы — 7 дней', fontsize=15, y=0.98)

colors = {
    'tAir':   '#58a6ff',
    'tCan':   '#3fb950',
    'tFlr':   '#f78166',
    'tPipe':  '#ff7b72',
    'tCovIn': '#d2a8ff',
    'tOut':   '#8b949e',
    'tSo1':   '#e3b341',
}

ax = axes[0]
ax.plot(df['dt'], df['tAir'],   color=colors['tAir'],   lw=1.8, label='Воздух (tAir)', zorder=5)
ax.plot(df['dt'], df['tCan'],   color=colors['tCan'],   lw=1.5, label='Полог (tCan)', zorder=4)
ax.plot(df['dt'], df['tCovIn'],color=colors['tCovIn'], lw=1.2, label='Кровля внутри (tCovIn)', alpha=0.8, zorder=3)
if 'tOut' in df.columns:
    ax.plot(df['dt'], df['tOut'], color=colors['tOut'], lw=1.0, ls='--', label='Снаружи (tOut)', alpha=0.7, zorder=2)
ax.axhline(18, color='#f78166', ls=':', lw=1, alpha=0.5, label='Уставка 18°C')
ax.set_ylabel('Температура, °C')
ax.legend(loc='upper right', ncol=2, fontsize=9)
ax.grid(True, alpha=0.4)
ax.set_title('Температуры воздуха и поверхностей')

ax = axes[1]
ax.plot(df['dt'], df['tPipe'],  color=colors['tPipe'], lw=1.8, label='Трубы отопления (tPipe)')
ax.plot(df['dt'], df['tFlr'],   color=colors['tFlr'],  lw=1.5, label='Пол (tFlr)')
ax.plot(df['dt'], df['tSo1'],   color=colors['tSo1'],  lw=1.2, label='Почва слой 1 (tSo1)', alpha=0.85)
ax.set_ylabel('Температура, °C')
ax.legend(loc='upper right', ncol=2, fontsize=9)
ax.grid(True, alpha=0.4)
ax.set_title('Отопление и тепловая инерция пола')

ax = axes[2]
if 'iGlob' in df.columns:
    ax2 = ax.twinx()
    ax2.fill_between(df['dt'], df['iGlob'], alpha=0.15, color='#e3b341', label='Солнечная радиация')
    ax2.set_ylabel('Солнечная радиация, Вт/м²', color='#e3b341')
    ax2.tick_params(axis='y', colors='#e3b341')
    ax2.set_ylim(0, df['iGlob'].max() * 3)
ax.plot(df['dt'], df['RH_air'], color='#79c0ff', lw=1.8, label='Относит. влажность')
ax.set_ylabel('Влажность, %')
ax.set_ylim(40, 105)
ax.axhline(85, color='#ff7b72', ls=':', lw=1, alpha=0.6, label='Граница конденсации 85%')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.4)
ax.set_title('Относительная влажность и солнечная радиация')

# Форматирование дат по оси X
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%H:%M'))
axes[2].xaxis.set_major_locator(mdates.DayLocator())

plt.tight_layout()
plt.savefig('../notebooks/climate_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 График сохранён в notebooks/climate_plot.png')

## 5. 🌿 Рост урожая (биомасса томатов)

Модель урожая — это **источно-поглотительная** модель:  
- **Источник**: фотосинтез → `cBuf` (буфер ассимилятов)  
- **Поглотители**: `cFruit`, `cLeaf`, `cStem` (распределение биомассы)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('🍅 Рост урожая томатов — распределение биомассы', fontsize=14, y=0.98)

# Конвертируем мг·CH₂O/м² → г/м²
scale = 1 / 1000

# 1. Биомасса по органам
ax = axes[0, 0]
ax.fill_between(df['dt'], df['cFruit']*scale, alpha=0.7, color='#ff7b72', label='Плоды')
ax.fill_between(df['dt'], df['cLeaf']*scale,  alpha=0.7, color='#3fb950', label='Листья')
ax.fill_between(df['dt'], df['cStem']*scale,  alpha=0.7, color='#e3b341', label='Стебель')
ax.set_ylabel('г·CH₂O / м²')
ax.set_title('Биомасса по органам')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))

# 2. Буфер ассимилятов
ax = axes[0, 1]
ax.plot(df['dt'], df['cBuf']*scale, color='#d2a8ff', lw=2, label='Буфер cBuf')
ax.fill_between(df['dt'], df['cBuf']*scale, alpha=0.3, color='#d2a8ff')
ax.axhline(20, color='#f78166', ls='--', lw=1.2, alpha=0.7, label='cBuf min (20 г/м²)')
ax.set_ylabel('г·CH₂O / м²')
ax.set_title('Буфер ассимилятов (резерв)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))

# 3. Структура биомассы (pie в конце симуляции)
ax = axes[1, 0]
final = df.iloc[-1]
sizes = [final['cFruit'], final['cLeaf'], final['cStem']]
labels = ['Плоды', 'Листья', 'Стебель']
colors_pie = ['#ff7b72', '#3fb950', '#e3b341']
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors_pie,
    autopct='%1.1f%%', startangle=90,
    textprops={'color': '#e6edf3'},
    wedgeprops={'edgecolor': '#0d1117', 'linewidth': 2}
)
for at in autotexts:
    at.set_color('#0d1117')
    at.set_fontweight('bold')
ax.set_title(f'Распределение биомассы\n(конец симуляции: {df["dt"].iloc[-1].strftime("%d.%m.%Y")})')

# 4. Температурный интеграл
ax = axes[1, 1]
ax.plot(df['dt'], df['tCanSum'], color='#58a6ff', lw=2, label='tCanSum (°C·сут)')
ax.fill_between(df['dt'], df['tCanSum'], alpha=0.3, color='#58a6ff')
ax.set_ylabel('°C · сут')
ax.set_title('Температурный интеграл культуры\n(определяет скорость развития)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))

plt.tight_layout()
plt.show()

## 6. 🔴 Концепция цифрового двойника: симуляция vs. датчики

Цифровой двойник непрерывно **сравнивает** симуляцию с данными реальных датчиков.  
Здесь мы **имитируем показания датчиков** добавлением реалистичного шума к симуляции.

In [ ]:
np.random.seed(42)

# ─── Параметры датчиков ──────────────────────────────────────────────────────
SENSOR_NOISE = {
    'tAir':   {'sigma': 0.3,  'bias': 0.1,  'unit': '°C',  'label': 'Температура воздуха'},
    'tCan':   {'sigma': 0.5,  'bias': -0.2, 'unit': '°C',  'label': 'Температура полога'},
    'RH_air': {'sigma': 2.0,  'bias': 1.5,  'unit': '%',   'label': 'Влажность'},
}
if 'co2_ppm' in df.columns:
    SENSOR_NOISE['co2_ppm'] = {'sigma': 20, 'bias': 10, 'unit': 'ppm', 'label': 'CO₂'}

sensors = {}
n = len(df)

# Имитация: белый шум + небольшой дрейф
for var, params in SENSOR_NOISE.items():
    white_noise = np.random.normal(0, params['sigma'], n)
    drift = np.linspace(0, params['bias'], n)          # медленный дрейф
    spike_idx = np.random.choice(n, size=3, replace=False)
    spikes = np.zeros(n)
    spikes[spike_idx] = np.random.normal(0, params['sigma']*4, 3)  # редкие выбросы
    sensors[var] = df[var].values + white_noise + drift + spikes

sensors_df = pd.DataFrame(sensors, index=df.index)
sensors_df['dt'] = df['dt']

# ─── Визуализация ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(SENSOR_NOISE), 1, figsize=(15, 4*len(SENSOR_NOISE)), sharex=True)
if len(SENSOR_NOISE) == 1:
    axes = [axes]

fig.suptitle('🔴 Цифровой двойник: Модель vs. Датчики', fontsize=15, y=1.01)

for i, (var, params) in enumerate(SENSOR_NOISE.items()):
    ax = axes[i]
    
    # Зона погрешности
    ax.fill_between(
        df['dt'],
        df[var] - 2*params['sigma'],
        df[var] + 2*params['sigma'],
        alpha=0.15, color='#58a6ff', label='Диапазон ±2σ модели'
    )
    # Симуляция
    ax.plot(df['dt'], df[var], color='#58a6ff', lw=2.0, label='🔵 Модель (GreenLight)', zorder=5)
    # Датчик
    ax.scatter(sensors_df['dt'][::4], sensors_df[var][::4],
               color='#f78166', s=8, alpha=0.6, label='🔴 Датчик (с шумом)', zorder=4)
    
    # Остатки
    residual = sensors_df[var].values - df[var].values
    rmse = np.sqrt(np.mean(residual**2))
    
    ax.set_ylabel(f'{params["unit"]}')
    ax.set_title(f'{params["label"]}  |  RMSE = {rmse:.2f} {params["unit"]}')
    ax.legend(loc='upper right', fontsize=9, ncol=3)
    ax.grid(True, alpha=0.4)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%H:%M'))
axes[-1].xaxis.set_major_locator(mdates.DayLocator())

plt.tight_layout()
plt.show()
print('\n📊 RMSE — мера расхождения модели и датчика (чем меньше — тем лучше)')

In [ ]:
# Анализ остатков
fig, axes = plt.subplots(1, len(SENSOR_NOISE), figsize=(5*len(SENSOR_NOISE), 5))
if len(SENSOR_NOISE) == 1:
    axes = [axes]

fig.suptitle('📊 Распределение ошибок (Модель − Датчик)', fontsize=13)

for i, (var, params) in enumerate(SENSOR_NOISE.items()):
    ax = axes[i]
    residual = sensors_df[var].values - df[var].values
    ax.hist(residual, bins=40, color='#58a6ff', edgecolor='#0d1117', alpha=0.8)
    ax.axvline(0, color='#3fb950', lw=2, label='Идеал')
    ax.axvline(np.mean(residual), color='#f78166', lw=1.5, ls='--', label=f'Смещение: {np.mean(residual):.2f}')
    ax.set_xlabel(f'Ошибка, {params["unit"]}')
    ax.set_ylabel('Частота')
    ax.set_title(params['label'])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 🎛 Интерактивная панель управления

Используйте ползунки для исследования **временного окна** и **сравнения переменных**.

In [ ]:
# Переменные доступные для просмотра
VARS_AVAILABLE = {
    'tAir':    ('Температура воздуха',        '°C',        '#58a6ff'),
    'tCan':    ('Температура полога',          '°C',        '#3fb950'),
    'tFlr':    ('Температура пола',            '°C',        '#f78166'),
    'tPipe':   ('Температура труб отопления',  '°C',        '#ff7b72'),
    'RH_air':  ('Относительная влажность',     '%',         '#79c0ff'),
    'cFruit':  ('Биомасса плодов',             'мг/м²',    '#ff7b72'),
    'cLeaf':   ('Биомасса листьев',            'мг/м²',    '#3fb950'),
    'cStem':   ('Биомасса стебля',             'мг/м²',    '#e3b341'),
    'cBuf':    ('Буфер ассимилятов',           'мг/м²',    '#d2a8ff'),
    'tCanSum': ('Темп. интеграл культуры',     '°C·сут',   '#58a6ff'),
    'tSo1':    ('Температура почвы (слой 1)',   '°C',        '#e3b341'),
    'vpAir':   ('Парциальное давление пара',   'Па',        '#79c0ff'),
}
if 'co2_ppm' in df.columns:
    VARS_AVAILABLE['co2_ppm'] = ('CO₂ концентрация', 'ppm', '#ffa657')
if 'iGlob' in df.columns:
    VARS_AVAILABLE['iGlob'] = ('Солнечная радиация', 'Вт/м²', '#e3b341')

var_options = [(f'{meta[0]} ({var})', var) for var, meta in VARS_AVAILABLE.items() 
               if var in df.columns]

out = widgets.Output()

w_var = widgets.Dropdown(
    options=var_options,
    value='tAir',
    description='Переменная:',
    layout=Layout(width='420px'),
    style={'description_width': '100px'}
)
w_day_start = widgets.IntSlider(
    value=0, min=0, max=6, step=1,
    description='День (нач):',
    layout=Layout(width='420px'),
    style={'description_width': '100px'}
)
w_day_end = widgets.IntSlider(
    value=7, min=1, max=7, step=1,
    description='День (кон):',
    layout=Layout(width='420px'),
    style={'description_width': '100px'}
)
w_show_sensor = widgets.Checkbox(
    value=True, description='Показать датчик',
    style={'description_width': '0px'}
)
w_show_band = widgets.Checkbox(
    value=True, description='Показать ±2σ',
    style={'description_width': '0px'}
)

def plot_interactive(var, day_start, day_end, show_sensor, show_band):
    with out:
        clear_output(wait=True)
        
        if var not in df.columns:
            print(f'❌ Переменная {var} не найдена')
            return
        
        t_start = t0 + timedelta(days=day_start)
        t_end   = t0 + timedelta(days=day_end)
        mask = (df['dt'] >= t_start) & (df['dt'] <= t_end)
        sub  = df[mask]

        if len(sub) == 0:
            print('⚠️ Нет данных в этом диапазоне')
            return

        meta = VARS_AVAILABLE.get(var, (var, '?', '#58a6ff'))
        label, unit, color = meta
        sigma_est = SENSOR_NOISE.get(var, {}).get('sigma', 0)

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), 
                                        gridspec_kw={'height_ratios': [3, 1]})
        fig.suptitle(f'📡 Цифровой двойник: {label}', fontsize=13)

        # Зона погрешности
        if show_band and sigma_est > 0:
            ax1.fill_between(
                sub['dt'], sub[var] - 2*sigma_est, sub[var] + 2*sigma_est,
                alpha=0.2, color=color, label=f'±2σ = ±{2*sigma_est:.1f} {unit}'
            )

        # Модель
        ax1.plot(sub['dt'], sub[var], color=color, lw=2.0, label='🔵 Модель', zorder=5)
        ax1.set_ylabel(f'{label} [{unit}]')
        ax1.grid(True, alpha=0.35)

        # Датчик
        if show_sensor and var in sensors_df.columns and var in SENSOR_NOISE:
            s_mask = (sensors_df['dt'] >= t_start) & (sensors_df['dt'] <= t_end)
            sub_s  = sensors_df[s_mask]
            ax1.scatter(sub_s['dt'][::2], sub_s[var][::2],
                        color='#f78166', s=10, alpha=0.7, label='🔴 Датчик', zorder=4)

            # Остатки
            residual = sub_s[var].values - sub[var].values[:len(sub_s)]
            ax2.plot(sub_s['dt'], residual, color='#f78166', lw=1.0, alpha=0.7)
            ax2.fill_between(sub_s['dt'], residual, alpha=0.2, color='#f78166')
            ax2.axhline(0, color='#3fb950', lw=1.5)
            ax2.axhline(sigma_est, color='#e3b341', ls='--', lw=1, alpha=0.7)
            ax2.axhline(-sigma_est, color='#e3b341', ls='--', lw=1, alpha=0.7)
            ax2.set_ylabel(f'Остаток [{unit}]')
            ax2.set_title(f'RMSE = {np.sqrt(np.mean(residual**2)):.3f} {unit}')
        else:
            ax2.text(0.5, 0.5, 'Нет данных датчика для этой переменной',
                     ha='center', va='center', transform=ax2.transAxes,
                     color='#8b949e', fontsize=10)

        ax2.grid(True, alpha=0.35)

        for ax in [ax1, ax2]:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m %H:%M'))
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=12))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

        # Статистика
        stats_text = (f'Среднее: {sub[var].mean():.2f} {unit}\n'
                      f'Мин: {sub[var].min():.2f}   Макс: {sub[var].max():.2f}')
        ax1.text(0.01, 0.02, stats_text, transform=ax1.transAxes,
                 fontsize=8, color='#8b949e', va='bottom',
                 bbox=dict(boxstyle='round', facecolor='#161b22', alpha=0.8, edgecolor='#30363d'))

        ax1.legend(loc='upper right', fontsize=9, ncol=3)
        plt.tight_layout()
        plt.show()

ui = VBox([
    widgets.HTML('<h4 style="color:#e6edf3">⚙️ Параметры визуализации</h4>'),
    w_var,
    HBox([w_day_start, w_day_end]),
    HBox([w_show_sensor, w_show_band]),
])

widgets.interactive_output(
    plot_interactive,
    {'var': w_var, 'day_start': w_day_start, 'day_end': w_day_end,
     'show_sensor': w_show_sensor, 'show_band': w_show_band}
)
display(ui, out)
plot_interactive('tAir', 0, 7, True, True)

## 8. 🔬 Анализ теплового баланса

Показываем, как тепло распределяется между элементами теплицы.

In [ ]:
# Выбираем 1 репрезентативный день
day3 = t0 + timedelta(days=2)
day4 = t0 + timedelta(days=3)
d_mask = (df['dt'] >= day3) & (df['dt'] <= day4)
sub = df[d_mask]

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('🔬 Детальный анализ: День 3 симуляции', fontsize=13, y=0.98)

# 1. Все слои температуры почвы
ax = axes[0, 0]
soil_colors = ['#ff7b72', '#f0883e', '#e3b341', '#3fb950', '#58a6ff']
for i, (col, c) in enumerate(zip(['tSo1','tSo2','tSo3','tSo4','tSo5'], soil_colors)):
    if col in sub.columns:
        ax.plot(sub['dt'], sub[col], color=c, lw=1.5, label=f'Слой {i+1} ({col})')
ax.set_ylabel('Температура, °C')
ax.set_title('Профиль температуры почвы (5 слоёв)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# 2. Сравнение tAir и tCan
ax = axes[0, 1]
ax.plot(sub['dt'], sub['tAir'], color='#58a6ff', lw=2, label='Воздух')
ax.plot(sub['dt'], sub['tCan'], color='#3fb950', lw=2, label='Полог')
ax.fill_between(sub['dt'], sub['tAir'], sub['tCan'], 
                where=(sub['tCan'] >= sub['tAir']),
                alpha=0.2, color='#3fb950', label='tCan > tAir (разогрев)')
ax.fill_between(sub['dt'], sub['tAir'], sub['tCan'],
                where=(sub['tCan'] < sub['tAir']),
                alpha=0.2, color='#58a6ff', label='tCan < tAir (охлаждение)')
ax.set_ylabel('Температура, °C')
ax.set_title('Разница воздух — полог')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# 3. Влажность
ax = axes[1, 0]
ax2 = ax.twinx()
ax.plot(sub['dt'], sub['vpAir'], color='#79c0ff', lw=2, label='vpAir (Па)')
ax2.plot(sub['dt'], sub['RH_air'], color='#d2a8ff', lw=1.5, ls='--', label='RH (%)')
ax2.axhline(85, color='#ff7b72', ls=':', lw=1, alpha=0.6)
ax.set_ylabel('Давление пара, Па', color='#79c0ff')
ax2.set_ylabel('Влажность, %', color='#d2a8ff')
ax.set_title('Влажность воздуха')
ax.tick_params(axis='y', colors='#79c0ff')
ax2.tick_params(axis='y', colors='#d2a8ff')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
ax.grid(True, alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# 4. Накопление биомассы
ax = axes[1, 1]
full_day = df.copy()
ax.stackplot(
    full_day['dt'],
    full_day['cFruit']/1000, full_day['cLeaf']/1000, full_day['cStem']/1000,
    labels=['Плоды', 'Листья', 'Стебель'],
    colors=['#ff7b72', '#3fb950', '#e3b341'],
    alpha=0.75
)
ax.set_ylabel('г·CH₂O / м²')
ax.set_title('Накопление биомассы (7 дней)')
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
ax.xaxis.set_major_locator(mdates.DayLocator())

plt.tight_layout()
plt.show()

## 9. 🎯 Что-если сценарии: Подключение физических данных

В реальном цифровом двойнике здесь подключаются **настоящие датчики**.  
Ниже показан шаблон для загрузки данных из CSV-файла вашей теплицы.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ШАБЛОН: Загрузка данных из ВАШЕЙ теплицы                       ║
# ║  Замените путь на реальный файл CSV с датчиков                  ║
# ╚══════════════════════════════════════════════════════════════════╝

SENSOR_CSV = None  # Замените: '/путь/к/данным_датчиков.csv'

# Ожидаемый формат CSV:
# timestamp,tAir,tCan,RH,co2,tPipe,iGlob
# 2024-01-01 00:00:00,18.5,19.1,75.3,850,45.2,0
# ...

if SENSOR_CSV and os.path.exists(SENSOR_CSV):
    physical_df = pd.read_csv(SENSOR_CSV, parse_dates=['timestamp'])
    print(f'✅ Загружено {len(physical_df)} измерений из {SENSOR_CSV}')
    print(physical_df.head())
else:
    print('ℹ️  Файл датчиков не указан. Используем синтетические данные.')
    print('   Чтобы подключить реальную теплицу:')
    print('   1. Экспортируйте данные SCADA/датчиков в CSV')
    print('   2. Укажите путь: SENSOR_CSV = "/путь/к/файлу.csv"')
    print('   3. Запустите ячейку снова')
    print()
    print('   Форматы поддерживаемых SCADA-систем:')
    print('   • Priva Connext  → экспорт CSV')
    print('   • Ridder Drive   → экспорт Excel → convert to CSV')
    print('   • Arduino/ESP32  → SD-карта CSV')
    print('   • MQTT broker    → influxdb → CSV export')

In [ ]:
# ══════════════════════════════════════════════════════════
# ФИНАЛЬНЫЙ ДАШБОРД: обзор всех ключевых переменных
# ══════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

fig.suptitle('🌱 Цифровой двойник теплицы — Сводная панель',
             fontsize=16, y=0.98, fontweight='bold')

# Вспомогательная функция
def mini_plot(ax, col, label, unit, color, fill=False):
    if col not in df.columns:
        ax.text(0.5, 0.5, f'{col}\nнедоступна', ha='center', va='center',
                transform=ax.transAxes, color='#8b949e')
        return
    ax.plot(df['dt'], df[col], color=color, lw=1.5)
    if fill:
        ax.fill_between(df['dt'], df[col], alpha=0.2, color=color)
    ax.set_title(label, fontsize=10)
    ax.set_ylabel(unit, fontsize=8)
    mn, mx = df[col].min(), df[col].max()
    ax.text(0.02, 0.95, f'{mn:.1f}…{mx:.1f}', transform=ax.transAxes,
            fontsize=7.5, color='#8b949e', va='top')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), fontsize=7)

mini_plot(fig.add_subplot(gs[0, 0]), 'tAir',   '🌡 Темп. воздуха',      '°C',    '#58a6ff')
mini_plot(fig.add_subplot(gs[0, 1]), 'tCan',   '🌿 Темп. полога',       '°C',    '#3fb950')
mini_plot(fig.add_subplot(gs[0, 2]), 'tPipe',  '🔥 Трубы отопления',    '°C',    '#ff7b72')
mini_plot(fig.add_subplot(gs[1, 0]), 'RH_air', '💧 Влажность',          '%',     '#79c0ff', fill=True)
mini_plot(fig.add_subplot(gs[1, 1]), 'vpAir',  '💨 Давление пара',      'Па',    '#d2a8ff')
if 'co2_ppm' in df.columns:
    mini_plot(fig.add_subplot(gs[1, 2]), 'co2_ppm','🌬 CO₂',             'ppm',   '#ffa657')
else:
    mini_plot(fig.add_subplot(gs[1, 2]), 'tFlr',  '🏗 Темп. пола',       '°C',    '#f78166')
mini_plot(fig.add_subplot(gs[2, 0]), 'cBuf',   '🔋 Буфер ассимил.',     'мг/м²', '#d2a8ff', fill=True)
mini_plot(fig.add_subplot(gs[2, 1]), 'cFruit', '🍅 Биомасса плодов',    'мг/м²', '#ff7b72', fill=True)
mini_plot(fig.add_subplot(gs[2, 2]), 'tCanSum','🌱 Темп. интеграл',     '°C·сут','#58a6ff', fill=True)

plt.savefig('../notebooks/dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('💾 Дашборд сохранён в notebooks/dashboard.png')

---

## 📚 Итог: Как цифровой двойник помогает?

| Задача | Как использовать |
|---|---|
| **Мониторинг** | Загрузите CSV с датчиков → сравните с моделью |
| **Диагностика** | Большие остатки → проблема с датчиком или системой |
| **Что-если** | Измените параметры модели → проверьте эффект без риска |
| **Оптимизация** | Найдите оптимальные уставки отопления/вентиляции |
| **Прогноз** | Запустите на прогнозе погоды → предскажите условия завтра |

### Следующие шаги:
1. 📡 **Подключить MQTT** — получать данные с датчиков в реальном времени  
2. 🔄 **Ассимиляция данных** — калибровать модель по реальным измерениям (EnKF)  
3. 🎛 **Оптимальное управление** — использовать модель для MPC-регулятора  
4. 🔔 **Алерты** — уведомления при отклонении датчиков от модели > 3σ  